In [ ]:
"""
Telecom Internal Knowledge Assistant — RAG + CRAG (Corrective RAG)
==================================================================
Demonstrates:
1. Building a mini telecom knowledge base (synthetic)
2. Standard RAG retrieval with telecom FAQs
3. CRAG — retrieval validation and correction flow
4. Side-by-side comparison of normal vs corrective retrieval

"""

In [ ]:
import os, sys, json, re, textwrap, hashlib, math
from collections import Counter
from dotenv import load_dotenv
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from colorama import init, Fore, Style
from tabulate import tabulate

In [ ]:
init(autoreset=True)
load_dotenv(os.path.join(os.path.dirname(__file__), "..", "key.env"))


In [ ]:
try:
    from openai import OpenAI
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))
except Exception:
    print(f"{Fore.YELLOW}⚠ OpenAI not available, falling back to mock mode{Style.RESET_ALL}")
    MOCK_MODE = True

In [ ]:
# ── Synthetic Telecom Knowledge Base ─────────────────────────────────────
KNOWLEDGE_BASE = [
    {
        "id": "SOP-001", "category": "Network Operations",
        "title": "5G NR Cell Outage Recovery SOP",
        "content": (
            "When a 5G NR cell outage is detected: 1) Verify alarm in NMS — check for RACH failure or S1 link down. "
            "2) SSH into gNodeB and run 'show cell-status'. 3) If hardware fault, dispatch field engineer with RRU replacement kit. "
            "4) If software fault, perform controlled restart via OSS. 5) Validate KPIs — RSRP > -110 dBm, SINR > 5 dB, "
            "and throughput within 80% baseline within 30 minutes post-recovery. Escalate to L3 if not resolved in 2 hours."
        ),
    },
    {
        "id": "SOP-002", "category": "Network Operations",
        "title": "LTE eNodeB Handover Failure Troubleshooting",
        "content": (
            "For LTE handover failures: 1) Check X2 link status between source and target eNodeBs. "
            "2) Verify neighbor cell list configuration — missing NCL entries cause 40% of handover failures. "
            "3) Analyze Measurement Report — ensure A3 event threshold is set correctly (typical offset: 3 dB). "
            "4) Check for timing advance issues if inter-frequency handover. "
            "5) Review HO success rate KPI — target > 98%. If persistent, check for antenna tilt misconfiguration."
        ),
    },
    {
        "id": "SOP-003", "category": "Customer Support",
        "title": "Fiber Broadband Speed Issue Resolution",
        "content": (
            "For fiber broadband speed complaints: 1) Run line test from OLT to ONT — check optical power levels "
            "(acceptable: -8 to -25 dBm). 2) Verify provisioned speed profile matches customer plan. "
            "3) Check ONT firmware version and update if outdated. 4) Test with direct Ethernet connection bypassing WiFi router. "
            "5) If speed is below 70% of provisioned rate, check for fiber bend loss or splitter issues. "
            "6) Escalate to fiber maintenance if optical loss exceeds 28 dB."
        ),
    },
    {
        "id": "POL-001", "category": "Compliance",
        "title": "SLA Penalty Calculation Policy — Enterprise Customers",
        "content": (
            "Enterprise SLA penalties are calculated as follows: Tier-1 (99.99% uptime): penalty = 10% monthly fee per hour "
            "of downtime exceeding SLA. Tier-2 (99.95%): penalty = 5% per hour. Tier-3 (99.9%): penalty = 2% per hour. "
            "Planned maintenance windows (announced 72h prior) are excluded. Penalties cap at 100% of monthly recurring charge. "
            "Force majeure events require VP-level approval for SLA credit. Updated January 2025, replaces POL-001-v3."
        ),
    },
    {
        "id": "POL-002", "category": "Compliance",
        "title": "Data Retention and Privacy Policy — Telecom",
        "content": (
            "Customer CDR (Call Detail Records) must be retained for 24 months per TRAI regulations. "
            "IP session logs retained for 12 months. Location data anonymized after 6 months. "
            "All PII must be encrypted at rest (AES-256) and in transit (TLS 1.3). "
            "Data access requires role-based authorization. Audit logs maintained for 36 months. "
            "Breach notification within 72 hours to CERT-In as per IT Act 2000 amendments."
        ),
    },
    {
        "id": "TSG-001", "category": "Troubleshooting",
        "title": "VoLTE Call Drop Troubleshooting Guide",
        "content": (
            "VoLTE call drops troubleshooting: 1) Check IMS registration status — verify P-CSCF and S-CSCF connectivity. "
            "2) Analyze SIP signaling — look for 408 Timeout or 503 Service Unavailable. "
            "3) Verify dedicated EPS bearer (QCI=1) is established and maintained during call. "
            "4) Check codec negotiation — AMR-WB preferred, AMR-NB fallback. "
            "5) Review RSRP at handover boundary — VoLTE requires > -115 dBm for stable calls. "
            "6) If inter-RAT SRVCC fallback fails, check 2G/3G coverage at drop location."
        ),
    },
    {
        "id": "TSG-002", "category": "Troubleshooting",
        "title": "MPLS VPN Circuit Provisioning Issues",
        "content": (
            "MPLS VPN provisioning troubleshooting: 1) Verify VRF configuration on PE router — check route distinguisher "
            "and route target import/export. 2) Ensure BGP peering between PE routers is established. "
            "3) Validate CE-PE routing protocol (OSPF/BGP/Static). 4) Check label switching path — "
            "run 'traceroute mpls' to verify LSP. 5) If partial connectivity, check for MTU mismatch (MPLS adds 4-byte label). "
            "6) Verify QoS policy mapping for enterprise traffic classes (EF, AF, BE)."
        ),
    },
    {
        "id": "KB-001", "category": "Knowledge Base",
        "title": "WiFi 6E Deployment Best Practices for Enterprise",
        "content": (
            "WiFi 6E (6 GHz band) enterprise deployment: 1) Conduct RF site survey for 6 GHz propagation — "
            "higher frequency means shorter range, plan for 20-30% more APs. 2) Use 160 MHz channels for max throughput. "
            "3) Enable WPA3-Enterprise with 802.1X. 4) Configure BSS coloring for interference management. "
            "5) Set minimum RSSI threshold to -67 dBm for reliable performance. "
            "6) Not all legacy devices support 6 GHz — maintain dual-band SSIDs for backward compatibility."
        ),
    },
    {
        "id": "KB-002", "category": "Knowledge Base",
        "title": "Billing System Integration — OCS/OFCS Architecture",
        "content": (
            "Online Charging System (OCS) handles prepaid real-time charging. Offline Charging (OFCS) processes "
            "postpaid CDRs. Integration points: 1) Diameter Ro interface for OCS (real-time credit control). "
            "2) Diameter Rf interface for OFCS (event-based charging). 3) Rating engine applies tariff plans. "
            "4) Balance management updates wallet in real-time. 5) Mediation platform normalizes CDR formats "
            "from network elements. 6) Revenue assurance module cross-checks rated CDRs against network events."
        ),
    },
    {
        "id": "SOP-004", "category": "Network Operations",
        "title": "Microwave Backhaul Link Degradation SOP",
        "content": (
            "Microwave link degradation procedure: 1) Check received signal level (RSL) — compare against threshold "
            "(typical: -65 dBm for 256-QAM). 2) If adaptive modulation active, check if link has downshifted "
            "(e.g., from 256-QAM to 16-QAM reduces capacity by 50%). 3) Verify antenna alignment — "
            "even 0.5 degree misalignment causes significant loss at 18+ GHz. 4) Check for rain fade events "
            "(especially E-band 70-80 GHz links). 5) Review BER — must be < 10^-6 for acceptable quality."
        ),
    },
]


In [ ]:
# ── Mock / Local Implementations ─────────────────────────────────────────
_tfidf_vectorizer = None
_tfidf_matrix = None

def _build_tfidf(kb):
    """Build TF-IDF index for local embedding simulation."""
    global _tfidf_vectorizer, _tfidf_matrix
    corpus = [f"{d['title']}. {d['content']}" for d in kb]
    _tfidf_vectorizer = TfidfVectorizer(stop_words="english", max_features=500)
    _tfidf_matrix = _tfidf_vectorizer.fit_transform(corpus).toarray()

def _tfidf_embed(text):
    """Convert text to TF-IDF vector."""
    return _tfidf_vectorizer.transform([text]).toarray()[0]

In [ ]:
# ── Embedding ────────────────────────────────────────────────────────────
_embedding_cache = {}

def get_embedding(text):
    if text in _embedding_cache:
        return _embedding_cache[text]
    if MOCK_MODE:
        emb = _tfidf_embed(text).tolist()
    else:
        resp = client.embeddings.create(input=[text], model="text-embedding-3-small")
        emb = resp.data[0].embedding
    _embedding_cache[text] = emb
    return emb


In [ ]:
def build_knowledge_index(kb):
    print(f"{Fore.CYAN}📚 Building knowledge base index ({len(kb)} documents)...{Style.RESET_ALL}")
    if MOCK_MODE:
        _build_tfidf(kb)
    for doc in kb:
        combined = f"{doc['title']}. {doc['content']}"
        doc["embedding"] = get_embedding(combined)
        print(f"  ✓ Indexed: {doc['id']} — {doc['title'][:50]}...")
    print(f"{Fore.GREEN}✅ Knowledge base indexed!{Style.RESET_ALL}\n")
    return kb

In [ ]:
# ── Standard RAG ─────────────────────────────────────────────────────────
def retrieve_top_k(query, kb, k=3):
    q_emb = np.array(get_embedding(query)).reshape(1, -1)
    results = []
    for doc in kb:
        d_emb = np.array(doc["embedding"]).reshape(1, -1)
        score = cosine_similarity(q_emb, d_emb)[0][0]
        results.append({**doc, "score": float(score)})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:k]

In [ ]:
def generate_answer(query, context_docs, system_note=""):
    context = "\n\n".join(f"[{d['id']}] {d['title']}\n{d['content']}" for d in context_docs)
    if MOCK_MODE:
        # Simple extractive answer in mock mode
        best = context_docs[0] if context_docs else None
        if best and best.get("id") == "WEB-FALLBACK":
            return best["content"]
        if best:
            note = f" ({system_note})" if system_note else ""
            return (
                f"Based on {best['id']} — {best['title']}:\n"
                f"{best['content'][:400]}\n"
                f"[Source: Internal Knowledge Base{note}]"
            )
        return "No relevant information found in the knowledge base."
    # OpenAI generation
    sys_prompt = (
        "You are a telecom internal knowledge assistant. Answer based ONLY on the provided context. "
        "If the context doesn't contain relevant information, say so clearly. Be concise and technical."
    )
    if system_note:
        sys_prompt += f"\n\nNote: {system_note}"
    resp = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
        temperature=0.2, max_tokens=500,
    )
    return resp.choices[0].message.content

In [ ]:
def standard_rag(query, kb):
    print(f"\n{Fore.YELLOW}{'='*70}")
    print(f"  STANDARD RAG PIPELINE")
    print(f"{'='*70}{Style.RESET_ALL}")
    print(f"{Fore.WHITE}Query: {query}{Style.RESET_ALL}\n")

    docs = retrieve_top_k(query, kb, k=3)
    print(f"{Fore.CYAN}Retrieved Documents:{Style.RESET_ALL}")
    for i, d in enumerate(docs, 1):
        print(f"  {i}. [{d['id']}] {d['title']} (score: {d['score']:.4f})")

    answer = generate_answer(query, docs)
    print(f"\n{Fore.GREEN}Answer:{Style.RESET_ALL}")
    for line in textwrap.wrap(answer, 80):
        print(f"  {line}")
    return {"query": query, "docs": docs, "answer": answer, "method": "Standard RAG"}

In [ ]:
# ── CRAG ──────────────────────────────────────────────────────────────────
def validate_retrieval(query, docs):
    if MOCK_MODE:
        # Keyword-overlap heuristic validation
        q_words = set(re.findall(r'\w+', query.lower()))
        best_score = docs[0]["score"] if docs else 0
        relevant_indices = []
        for i, d in enumerate(docs):
            d_words = set(re.findall(r'\w+', (d["title"] + " " + d["content"]).lower()))
            overlap = len(q_words & d_words) / max(len(q_words), 1)
            if overlap > 0.25 and d["score"] > 0.05:
                relevant_indices.append(i + 1)

        if best_score > 0.25 and len(relevant_indices) >= 1:
            verdict, conf = "CORRECT", min(0.95, best_score + 0.3)
            reason = f"Top doc has strong relevance (score={best_score:.2f}), keyword overlap confirms match."
        elif best_score > 0.08:
            verdict, conf = "AMBIGUOUS", best_score + 0.15
            reason = f"Moderate relevance (score={best_score:.2f}). Query may need refinement for better match."
        else:
            verdict, conf = "INCORRECT", best_score
            reason = f"Low relevance (score={best_score:.2f}). Topic likely not covered in knowledge base."
        return {"verdict": verdict, "confidence": round(conf, 2),
                "reasoning": reason, "relevant_doc_indices": relevant_indices or [1]}

    # OpenAI-based validation
    doc_summaries = "\n".join(
        f"Doc {i+1} [{d['id']}]: {d['title']} (similarity: {d['score']:.4f})\nContent: {d['content'][:150]}..."
        for i, d in enumerate(docs)
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "You are a retrieval quality evaluator for a telecom knowledge system. "
                "Evaluate whether the retrieved documents are relevant to the query. "
                'Respond in JSON: {"verdict": "CORRECT|AMBIGUOUS|INCORRECT", "confidence": 0.0-1.0, '
                '"reasoning": "brief explanation", "relevant_doc_indices": [1-indexed list]}'
            )},
            {"role": "user", "content": f"Query: {query}\n\nRetrieved:\n{doc_summaries}"},
        ],
        temperature=0.1, max_tokens=300, response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

In [ ]:
def refine_query(original_query, validation):
    if MOCK_MODE:
        # Rule-based refinement with telecom synonyms
        synonyms = {
            "5g": "5G NR gNodeB",
            "not working": "cell outage failure troubleshooting",
            "slow": "throughput degradation",
            "dropping": "call drop handover failure",
            "enb": "eNodeB LTE",
            "calls": "VoLTE call drop SRVCC",
            "speed": "throughput bandwidth",
            "internet": "broadband fiber data",
            "down": "outage alarm fault",
        }
        refined = original_query
        for k, v in synonyms.items():
            if k in original_query.lower():
                refined += f" {v}"
        return refined.strip()

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "You are a telecom query optimizer. Rewrite the query using precise telecom "
                "terminology (3GPP, ITU-T). Return ONLY the rewritten query."
            )},
            {"role": "user", "content": (
                f"Original: {original_query}\nIssue: {validation.get('reasoning', '')}\nRewrite:"
            )},
        ],
        temperature=0.3, max_tokens=150,
    )
    return resp.choices[0].message.content.strip()

In [ ]:
def web_search_fallback(query):
    if MOCK_MODE:
        return [{
            "id": "WEB-FALLBACK", "category": "External",
            "title": "External Knowledge (Web Search Fallback)",
            "content": (
                f"[Simulated Web Search Result for: '{query}']\n"
                "The internal knowledge base did not contain relevant information for this query. "
                "In a production system, this would trigger a web search via APIs (e.g., Tavily, "
                "Google Search) or query a structured external database (3GPP specs, vendor portals). "
                "The CRAG system correctly identified that internal retrieval was insufficient and "
                "activated the fallback mechanism to prevent hallucination."
            ),
            "score": 0.0,
        }]

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "You are a telecom expert. The internal KB couldn't answer this query. "
                "Provide a helpful answer based on general telecom knowledge. Be brief and technical."
            )},
            {"role": "user", "content": query},
        ],
        temperature=0.3, max_tokens=400,
    )
    return [{
        "id": "WEB-FALLBACK", "category": "External",
        "title": "External Knowledge (Web Search Fallback)",
        "content": resp.choices[0].message.content, "score": 0.0,
    }]

In [ ]:
def crag_pipeline(query, kb, max_retries=2):
    print(f"\n{Fore.MAGENTA}{'='*70}")
    print(f"  CRAG (CORRECTIVE RAG) PIPELINE")
    print(f"{'='*70}{Style.RESET_ALL}")
    print(f"{Fore.WHITE}Query: {query}{Style.RESET_ALL}\n")

    current_query = query
    attempt = 0
    history = []

    while attempt <= max_retries:
        attempt += 1
        print(f"{Fore.CYAN}── Attempt {attempt} ──{Style.RESET_ALL}")
        if current_query != query:
            print(f"{Fore.YELLOW}  Refined Query: {current_query}{Style.RESET_ALL}")

        docs = retrieve_top_k(current_query, kb, k=3)
        print(f"  Retrieved:")
        for i, d in enumerate(docs, 1):
            print(f"    {i}. [{d['id']}] {d['title']} (score: {d['score']:.4f})")

        print(f"\n  {Fore.YELLOW}⚖️  Validating retrieval...{Style.RESET_ALL}")
        validation = validate_retrieval(current_query, docs)
        verdict = validation.get("verdict", "INCORRECT").upper()
        confidence = validation.get("confidence", 0.0)
        reasoning = validation.get("reasoning", "")

        color = {"CORRECT": Fore.GREEN, "AMBIGUOUS": Fore.YELLOW, "INCORRECT": Fore.RED}.get(verdict, Fore.RED)
        print(f"  {color}Verdict: {verdict} (confidence: {confidence:.2f}){Style.RESET_ALL}")
        print(f"  Reasoning: {reasoning}")

        history.append({
            "attempt": attempt, "query": current_query, "verdict": verdict,
            "confidence": confidence, "docs": [d["id"] for d in docs],
        })

        if verdict == "CORRECT":
            rel_idx = validation.get("relevant_doc_indices", [1, 2, 3])
            filtered = [docs[i-1] for i in rel_idx if 0 < i <= len(docs)] or docs
            answer = generate_answer(query, filtered, "Validated by CRAG retrieval evaluator.")
            print(f"\n  {Fore.GREEN}✅ CORRECT — Generating from validated docs{Style.RESET_ALL}")
            print(f"\n{Fore.GREEN}Answer:{Style.RESET_ALL}")
            for line in textwrap.wrap(answer, 80):
                print(f"  {line}")
            return {"query": query, "docs": filtered, "answer": answer,
                    "method": "CRAG", "verdict": verdict, "attempts": attempt, "history": history}

        elif verdict == "AMBIGUOUS" and attempt <= max_retries:
            print(f"\n  {Fore.YELLOW}🔄 AMBIGUOUS — Refining query...{Style.RESET_ALL}")
            current_query = refine_query(current_query, validation)
            continue

        else:
            print(f"\n  {Fore.RED}🌐 INCORRECT — Falling back to external search...{Style.RESET_ALL}")
            fallback_docs = web_search_fallback(query)
            answer = fallback_docs[0]["content"]
            print(f"\n{Fore.GREEN}Answer (external fallback):{Style.RESET_ALL}")
            for line in textwrap.wrap(answer, 80):
                print(f"  {line}")
            return {"query": query, "docs": fallback_docs, "answer": answer,
                    "method": "CRAG", "verdict": "INCORRECT-FALLBACK",
                    "attempts": attempt, "history": history}

    return {"query": query, "docs": [], "answer": "Unable to resolve.",
            "method": "CRAG", "verdict": "FAILED", "attempts": attempt, "history": history}


In [ ]:
# ── Comparison ────────────────────────────────────────────────────────────
def run_comparison(query, kb):
    print(f"\n\n{'#'*70}")
    print(f"  COMPARISON: Standard RAG vs CRAG")
    print(f"  Query: \"{query}\"")
    print(f"{'#'*70}")

    rag_result = standard_rag(query, kb)
    crag_result = crag_pipeline(query, kb)

    print(f"\n{Fore.CYAN}{'='*70}")
    print(f"  COMPARISON SUMMARY")
    print(f"{'='*70}{Style.RESET_ALL}")

    table = [
        ["Method", rag_result["method"], crag_result["method"]],
        ["Docs Used", ", ".join(d["id"] for d in rag_result["docs"]),
         ", ".join(d["id"] for d in crag_result["docs"])],
        ["Validation", "None (blind)", crag_result.get("verdict", "N/A")],
        ["Attempts", "1", str(crag_result.get("attempts", 1))],
    ]
    print(tabulate(table, headers=["Dimension", "Standard RAG", "CRAG"], tablefmt="fancy_grid"))
    return rag_result, crag_result

In [ ]:
# ── Main ──────────────────────────────────────────────────────────────────
def main():
    mode = "MOCK (Local TF-IDF)" if MOCK_MODE else "LIVE (OpenAI API)"
    print(f"{Fore.MAGENTA}")
    print("╔══════════════════════════════════════════════════════════════════╗")
    print("║  Telecom Internal Knowledge Assistant — RAG + CRAG Demo        ║")
    print("║  SOPs · Troubleshooting · Policies · Compliance               ║")
    print(f"║  Mode: {mode:<55} ║")
    print("╚══════════════════════════════════════════════════════════════════╝")
    print(f"{Style.RESET_ALL}")

    kb = build_knowledge_index(KNOWLEDGE_BASE)

    test_queries = [
        {"query": "How do I troubleshoot VoLTE call drops?",
         "desc": "Direct match — expects TSG-001"},
        {"query": "my 5G is not working properly",
         "desc": "Vague query — CRAG should refine"},
        {"query": "What are the SLA penalty rules for enterprise downtime?",
         "desc": "Policy query — expects POL-001"},
        {"query": "How to configure Kubernetes pods for network function virtualization?",
         "desc": "Out-of-scope — CRAG should fallback"},
        {"query": "eNB dropping calls at cell edge with low RSRP",
         "desc": "Jargon-heavy — tests alignment"},
    ]

    print(f"\n{Fore.WHITE}Running {len(test_queries)} test scenarios...{Style.RESET_ALL}\n")
    all_results = []
    for i, tq in enumerate(test_queries, 1):
        print(f"\n{'━'*70}")
        print(f"{Fore.WHITE}  Scenario {i}: {tq['desc']}{Style.RESET_ALL}")
        print(f"{'━'*70}")
        rag_r, crag_r = run_comparison(tq["query"], kb)
        all_results.append({"scenario": tq["desc"], "rag": rag_r, "crag": crag_r})

    # Final summary
    print(f"\n\n{'█'*70}")
    print(f"{Fore.MAGENTA}  FINAL SUMMARY — ALL SCENARIOS{Style.RESET_ALL}")
    print(f"{'█'*70}\n")

    summary = []
    for i, r in enumerate(all_results, 1):
        summary.append([
            f"Q{i}", r["scenario"][:40],
            ", ".join(d["id"] for d in r["rag"]["docs"]),
            "Blind",
            ", ".join(d["id"] for d in r["crag"]["docs"]),
            r["crag"].get("verdict", "N/A"),
            r["crag"].get("attempts", 1),
        ])

    print(tabulate(summary,
                   headers=["#", "Scenario", "RAG Docs", "RAG Check", "CRAG Docs", "CRAG Verdict", "Attempts"],
                   tablefmt="fancy_grid"))

    print(f"\n{Fore.GREEN}✅ Demo complete! CRAG adds retrieval validation, query refinement,")
    print(f"   and external fallback — critical for telecom NOC/fault management.{Style.RESET_ALL}\n")


In [ ]:
main()